### Feature Engineering

#### Corrección de datos

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Raíz del proyecto calculada de forma reproducible
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = BASE_DIR / "data" / "processed" / "train.parquet"

df_train = pd.read_parquet(DATA_PATH)
print(f"Dataset cargado: {df_train.shape[0]} filas, {df_train.shape[1]} columnas")
df_train.head()

Dataset cargado: 3201 filas, 13 columnas


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1779,2,Graduate,Yes,1600000,4700000,12,582,1500000,1300000,5200000,1000000,Approved
1,486,4,Graduate,No,1600000,4900000,8,685,3200000,2000000,3800000,1500000,Approved
2,1092,4,Not Graduate,Yes,600000,2100000,6,558,1300000,1000000,2200000,300000,Approved
3,1912,1,Graduate,No,7500000,16000000,10,541,16300000,4500000,22400000,5100000,Rejected
4,3543,1,Graduate,No,6500000,23400000,2,672,5000000,10400000,16700000,7600000,Approved


One-Hot sobre categóricas (incluida el target)

In [2]:
# Limpieza preventiva de espacios en blanco en nombres de columnas y textos
df_train.columns = df_train.columns.str.strip()
str_cols = df_train.select_dtypes(include="object").columns
df_train[str_cols] = df_train[str_cols].apply(lambda col: col.str.strip())

# Mapeo binario explícito (0 y 1)
df_train["education"] = df_train["education"].map({"Not Graduate": 0, "Graduate": 1})
df_train["self_employed"] = df_train["self_employed"].map({"No": 0, "Yes": 1})
df_train["loan_status"] = df_train["loan_status"].map({"Rejected": 0, "Approved": 1})

# Verificación rápida de nulos y tipos
print(df_train[["education", "self_employed", "loan_status"]].isnull().sum())
df_train[["education", "self_employed", "loan_status"]].head()

education        0
self_employed    0
loan_status      0
dtype: int64


,education,self_employed,loan_status
0,1,1,1
1,1,0,1
2,0,1,1
3,1,0,0
4,1,0,1


Corrección de outliers

In [3]:
# Columnas patrimoniales que por definición de negocio deben ser >= 0
asset_cols = [
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value"
]

# Aplicamos clipping inferior en 0
df_train[asset_cols] = df_train[asset_cols].clip(lower=0)

# Verificación rápida del valor mínimo
print("Valores mínimos post-clipping:")
print(df_train[asset_cols].min())

Valores mínimos post-clipping:
residential_assets_value         0
commercial_assets_value          0
luxury_assets_value         300000
bank_asset_value                 0
dtype: int64


#### Creación y selección de variables

In [4]:
# Total de patrimonio neto en activos
df_train["total_assets"] = (
    df_train["residential_assets_value"]
    + df_train["commercial_assets_value"]
    + df_train["luxury_assets_value"]
    + df_train["bank_asset_value"]
)

# Ratios financieros clave (añadimos 0.05 al denominador para evitar división por cero)
# Cobertura de activos frente al préstamo
df_train["asset_to_loan_ratio"] = df_train["total_assets"] / (df_train["loan_amount"] + 0.05)

# Ratio de esfuerzo: Préstamo vs Ingreso Anual
df_train["loan_to_income_ratio"] = df_train["loan_amount"] / (df_train["income_annum"] + 0.05)

print(df_train[["total_assets", "asset_to_loan_ratio", "loan_to_income_ratio"]].describe())

       total_assets  asset_to_loan_ratio  loan_to_income_ratio
count  3.201000e+03          3201.000000           3201.000000
mean   3.229725e+07             2.237789              2.976564
std    1.946128e+07             0.641274              0.591570
min    5.000000e+05             0.750000              1.500000
25%    1.620000e+07             1.776596              2.470588
50%    3.100000e+07             2.149466              2.980000
75%    4.660000e+07             2.621053              3.479592
max    8.550000e+07             5.666666              4.000000


In [5]:
# Carga de pago anual sobre ingreso (proxy de DTI)
df_train["annual_payment_burden"] = ((df_train["loan_amount"] / df_train["loan_term"]) / (df_train["income_annum"] + 0.05))

# Ratio de liquidez inmediata (efectivo en banco vs préstamo solicitado)
df_train["liquidity_to_loan_ratio"] = (df_train["bank_asset_value"] / (df_train["loan_amount"] + 0.05))

# Ingreso ponderado por dependientes
df_train["income_per_dependent"] = (df_train["income_annum"] / (df_train["no_of_dependents"] + 0.05))

print(df_train[["annual_payment_burden", "liquidity_to_loan_ratio", "income_per_dependent"]].describe())

       annual_payment_burden  liquidity_to_loan_ratio  income_per_dependent
count            3201.000000              3201.000000          3.201000e+03
mean                0.436928                 0.339197          1.889900e+07
std                 0.397080                 0.130205          4.366570e+07
min                 0.092593                 0.000000          3.960396e+04
25%                 0.188816                 0.236842          1.037037e+06
50%                 0.276786                 0.326180          2.000000e+06
75%                 0.515152                 0.423469          4.780488e+06
max                 2.000000                 0.818182          1.980000e+08


Selección de variables

In [6]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif

# Eliminación de identificador (no aporta señal predictiva y sesga el modelo)
df_train = df_train.drop(columns=["loan_id"], errors="ignore")

# Matriz de correlación para detectar redundancias directas (|r| > 0.85)
corr_matrix = df_train.corr(numeric_only=True)
high_corr = np.where(np.abs(corr_matrix) > 0.85)
redundant_pairs = [
    (corr_matrix.index[x], corr_matrix.columns[y], round(corr_matrix.iloc[x, y], 3))
    for x, y in zip(*high_corr)
    if x != y and x < y
]

print("Pares de variables con alta colinealidad (|r| > 0.85):")
print(redundant_pairs if redundant_pairs else "Ninguno detectado.")

# Información Mutua respecto a la variable objetivo (loan_status)
features = [col for col in df_train.columns if col not in ["loan_status"]]
X = df_train[features]
y = df_train["loan_status"]

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=features).sort_values(ascending=False)

print("\nRanking de Información Mutua respecto a loan_status:")
print(mi_series.round(4))

Pares de variables con alta colinealidad (|r| > 0.85):
[('income_annum', 'loan_amount', np.float64(0.929)), ('income_annum', 'luxury_assets_value', np.float64(0.931)), ('income_annum', 'bank_asset_value', np.float64(0.85)), ('income_annum', 'total_assets', np.float64(0.933)), ('loan_amount', 'luxury_assets_value', np.float64(0.867)), ('loan_amount', 'total_assets', np.float64(0.87)), ('luxury_assets_value', 'total_assets', np.float64(0.931))]

Ranking de Información Mutua respecto a loan_status:
cibil_score                 0.4982
annual_payment_burden       0.0429
loan_term                   0.0355
no_of_dependents            0.0212
asset_to_loan_ratio         0.0183
luxury_assets_value         0.0157
total_assets                0.0077
residential_assets_value    0.0056
commercial_assets_value     0.0020
education                   0.0000
self_employed               0.0000
income_annum                0.0000
loan_amount                 0.0000
bank_asset_value            0.0000
loan_to_i

Por conociemiento de dominio, la poca cantidad de variables totales y la relación entre ellas, se elige un conjunto de variables manualmente.

In [7]:
# Definición explícita de las variables predictoras seleccionadas y el target
SELECTED_FEATURES = [
    "cibil_score",
    "annual_payment_burden",
    "loan_term",
    "asset_to_loan_ratio",
    "no_of_dependents",
    "loan_to_income_ratio"
]
TARGET_COL = "loan_status"

# Separación de matriz de características (X) y vector objetivo (y)
X_train = df_train[SELECTED_FEATURES].copy()
y_train = df_train[TARGET_COL].copy()

# Verificación de integridad
print(f"Dimensiones de X_train: {X_train.shape} (muestras, features)")
print(f"Dimensiones de y_train: {y_train.shape}")
print(f"\nDistribución del target (proporción de clases):")
print(y_train.value_counts(normalize=True).round(3))

# Vista previa de las entradas que verá el clasificador
X_train.head()

Dimensiones de X_train: (3201, 6) (muestras, features)
Dimensiones de y_train: (3201,)

Distribución del target (proporción de clases):
loan_status
1    0.622
0    0.378
Name: proportion, dtype: float64


,cibil_score,annual_payment_burden,loan_term,asset_to_loan_ratio,no_of_dependents,loan_to_income_ratio
0,582,0.244792,12,1.914894,2,2.937500
1,685,0.382812,8,2.142857,4,3.062500
2,558,0.583333,6,2.285714,4,3.500000
3,541,0.213333,10,3.018750,1,2.133333
4,672,1.800000,2,1.696581,1,3.600000


Escalado y versionado final del preprocesamiento

In [8]:
import joblib
from sklearn.preprocessing import RobustScaler

# Instanciamos y ajustamos el escalador con las variables predictoras (X_train)
scaler = RobustScaler()
scaler.set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)

# Guardamos el escalador ajustado en disco
joblib.dump(scaler, BASE_DIR / "models" / "scaler.joblib")
print("Escalador guardado en models/scaler.joblib")

Escalador guardado en models/scaler.joblib


In [9]:
# Guardamos la matriz sin escalado para evitar data leakage en el siguiente notebook
df_train_processed = X_train.copy()
df_train_processed["loan_status"] = y_train

df_train_processed.to_parquet(BASE_DIR / "data" / "processed" / "train_features.parquet", index=False)
print("Datos procesados guardados en data/processed/train_features.parquet")

Datos procesados guardados en data/processed/train_features.parquet
